# Week 6 — NonLinLoc setup: convert PyOcto output, prepare velocity model
Two parts: (1) convert PyOcto's preliminary event assignments into NonLinLoc's
.obs phase format, (2) inspect the real Chow et al. (2022) velocity model file
once downloaded, before building the Vel2Grid3D control file against its
actual structure rather than assumptions.

In [ ]:
import os
import numpy as np
import pandas as pd
import pyocto
import obspy

PROJECT_DIR = os.path.expanduser("~/tvz_project")
CATALOG_DIR = os.path.join(PROJECT_DIR, "catalogs")
NLL_DIR = os.path.join(PROJECT_DIR, "nonlinloc")
INVENTORY_FILE = os.path.join(PROJECT_DIR, "data", "inventory", "NZ_2019-08-01_2019-09-30.xml")
os.makedirs(os.path.join(NLL_DIR, "obs"), exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, "velocity_models"), exist_ok=True)

assignments = pd.read_csv(os.path.join(CATALOG_DIR, "eqtransformer_pyocto_assignments.csv"))
print(f"{len(assignments)} pick assignments loaded, columns: {list(assignments.columns)}")

In [ ]:
# to_nonlinloc is a staticmethod (confirmed from source), callable directly
# on the class without needing an associator instance.
obs_path = os.path.join(NLL_DIR, "obs", "eqtransformer_pyocto.obs")

pyocto.associator.OctoAssociator.to_nonlinloc(assignments, obs_path, pick_std=0.05)

print(f"Saved: {obs_path}")
with open(obs_path) as f:
    preview = f.readlines()[:15]
print("".join(preview))

In [ ]:
# Inspect both candidate files before deciding which (or both) to use.
# Confirmed variable naming ("vp", "vs") from IRIS EMC's own documented
# xarray usage example for this exact model.
import xarray as xr

VELMODEL_DIR = os.path.join(PROJECT_DIR, "velocity_models")

for fname in ["nz-atom-north-chow-etal-2021-vp+vs-shallow.r0.1-n4.nc",
              "nz-atom-north-chow-etal-2021-vp+vs-crust.r0.1-n4.nc"]:
    path = os.path.join(VELMODEL_DIR, fname)
    ds = xr.open_dataset(path)
    print(f"=== {fname} ===")
    print(ds)
    print()


In [ ]:
# Extract a real 1D depth profile from Chow et al. (2022) at Lake Taupo,
# using the provided 2D latitude/longitude arrays to find the nearest grid
# cell (sidesteps needing to know the underlying projected x/y coordinate
# system at all).
TARGET_LAT, TARGET_LON = -38.8, 176.0

def nearest_profile(ds, target_lat, target_lon):
    dist2 = (ds["latitude"] - target_lat) ** 2 + (ds["longitude"] - target_lon) ** 2
    iy, ix = np.unravel_index(np.argmin(dist2.values), dist2.shape)
    actual_lat = float(ds["latitude"][iy, ix])
    actual_lon = float(ds["longitude"][iy, ix])
    profile = ds.isel(y=iy, x=ix)
    return profile, actual_lat, actual_lon

shallow = xr.open_dataset(os.path.join(VELMODEL_DIR, "nz-atom-north-chow-etal-2021-vp+vs-shallow.r0.1-n4.nc"))
crust = xr.open_dataset(os.path.join(VELMODEL_DIR, "nz-atom-north-chow-etal-2021-vp+vs-crust.r0.1-n4.nc"))

shallow_profile, lat1, lon1 = nearest_profile(shallow, TARGET_LAT, TARGET_LON)
crust_profile, lat2, lon2 = nearest_profile(crust, TARGET_LAT, TARGET_LON)
print(f"Shallow file nearest grid cell: {lat1:.3f}, {lon1:.3f}")
print(f"Crust file nearest grid cell:   {lat2:.3f}, {lon2:.3f}")

# Combine: shallow model for depths < 7 km, crust model for depths >= 7 km
# (the two files overlap 7-8 km; splitting at 7 km avoids double-counting)
shallow_df = shallow_profile[["vp", "vs", "rho"]].to_dataframe().reset_index()
shallow_df = shallow_df[shallow_df["depth"] < 7.0]
crust_df = crust_profile[["vp", "vs", "rho"]].to_dataframe().reset_index()
crust_df = crust_df[crust_df["depth"] <= 40.0]  # clip to our target grid depth range

profile = pd.concat([shallow_df, crust_df], ignore_index=True).sort_values("depth").reset_index(drop=True)
print(f"\nCombined profile: {len(profile)} depth levels, {profile['depth'].min()} to {profile['depth'].max()} km")
print(profile.head(10))
print("...")
print(profile.tail(5))
print(f"\nrho range: {profile['rho'].min():.1f} to {profile['rho'].max():.1f} "
      f"-- if this is ~1.5-3.5, it's g/cm3 and needs x1000 for NonLinLoc's kg/m3; "
      f"if it's ~1500-3500, it's already kg/m3")


In [ ]:
# Generate the NonLinLoc control file for Vel2Grid (LAYER-based 1D model).
# Grid spans our full station/seismicity region (~300km) around the same
# reference point (Lake Taupo) used throughout the project, at 1km spacing,
# matching the vertical resolution of the real Chow et al. (2022) data.
#
# FIX 1: gridType changed from VELOCITY to SLOW_LEN - the Podvin-Lecomte
# travel-time method (GT_PLFD) specifically requires this grid type; real
# example control files confirm this ("VGGRID ... SLOW_LEN").
#
# FIX 2: top of grid extended from -2.0 to -3.0 km. Four Ruapehu-flank
# stations (FWVZ, MAVZ, TRVZ, WHVZ) sit at elevations the original grid
# didn't reach - MAVZ specifically needs -2.624 km, deeper than even the
# raw data's shallowest level (-2.25 km). No real tomography data exists
# above -2.25 km, so the extra layer extrapolates the shallowest real
# vp/vs/rho values upward as a reasonable approximation - noted for the
# methods section.
NLL_MODEL_DIR = os.path.join(NLL_DIR, "model")
os.makedirs(NLL_MODEL_DIR, exist_ok=True)

RHO_NEEDS_CONVERSION = profile["rho"].max() < 10
rho_multiplier = 1000.0 if RHO_NEEDS_CONVERSION else 1.0
print(f"rho conversion factor applied: x{rho_multiplier}")

control_lines = [
    "CONTROL 1 54321",
    f"TRANS SIMPLE {TARGET_LAT} {TARGET_LON} 0.0",
    "",
    f"VGOUT {NLL_MODEL_DIR}/tvz_layer",
    "VGTYPE P",
    "VGTYPE S",
    "VGGRID 301 301 44 -150.0 -150.0 -3.0 1.0 1.0 1.0 SLOW_LEN",
]

# Extrapolated padding layer at -3.0 km, using the shallowest real values (-2.25 km)
top_row = profile.iloc[0]
pad_rho = top_row["rho"] * rho_multiplier
control_lines.append(f"LAYER -3.000 {top_row['vp']:.4f} 0.0 {top_row['vs']:.4f} 0.0 {pad_rho:.1f} 0.0")

for _, row in profile.iterrows():
    depth = row["depth"]
    vp = row["vp"]
    vs = row["vs"]
    rho = row["rho"] * rho_multiplier
    control_lines.append(f"LAYER {depth:.3f} {vp:.4f} 0.0 {vs:.4f} 0.0 {rho:.1f} 0.0")

control_path = os.path.join(NLL_DIR, "vel2grid_tvz.in")
with open(control_path, "w") as f:
    f.write("\n".join(control_lines) + "\n")

print(f"Saved: {control_path}")
print(f"{len(profile) + 1} LAYER statements written (including the extrapolated top layer)")
print("\nFirst few lines:")
print("\n".join(control_lines[:10]))


In [ ]:
# Build GTSRCE lines using the exact station labels already used in the
# .obs file (from assignments["station"], e.g. "NZ.ARHZ.10") - guarantees
# they match, rather than reconstructing the label separately and risking
# a mismatch against a station with multiple location codes.
inventory = obspy.read_inventory(INVENTORY_FILE)

# lookup: "NZ.ARHZ" -> (lat, lon, elevation_km)
station_coords = {}
for net in inventory:
    for sta in net:
        key = f"{net.code}.{sta.code}"
        station_coords[key] = (sta.latitude, sta.longitude, sta.elevation / 1000.0)

unique_labels = sorted(assignments["station"].unique())
print(f"{len(unique_labels)} unique station labels in the assignments/obs data")

gtsrce_lines = []
missing = []
for label in unique_labels:
    parts = label.split(".")
    lookup_key = f"{parts[0]}.{parts[1]}"  # network.station, ignoring location code
    if lookup_key not in station_coords:
        missing.append(label)
        continue
    lat, lon, elev_km = station_coords[lookup_key]
    gtsrce_lines.append(f"GTSRCE {label} LATLON {lat:.5f} {lon:.5f} 0.0 {elev_km:.4f}")

if missing:
    print(f"WARNING: {len(missing)} labels had no inventory match: {missing}")
else:
    print("All station labels matched to inventory coordinates")

print(f"\n{len(gtsrce_lines)} GTSRCE lines built")
print("\n".join(gtsrce_lines[:5]))

In [ ]:
# Build Grid2Time control files. Based on real-world examples, GTFILES
# needs a separate run per wave type (unlike VGTYPE in Vel2Grid, which was
# repeatable in one run) - so we generate two control files, one for P one
# for S, sharing the same CONTROL/TRANS/GTSRCE statements.
NLL_TIME_DIR = os.path.join(NLL_DIR, "time")
os.makedirs(NLL_TIME_DIR, exist_ok=True)

vel_grid_root = os.path.join(NLL_MODEL_DIR, "tvz_layer")
time_grid_root = os.path.join(NLL_TIME_DIR, "tvz_layer")

shared_lines = [
    "CONTROL 1 54321",
    f"TRANS SIMPLE {TARGET_LAT} {TARGET_LON} 0.0",
    "",
] + gtsrce_lines + [""]

for wave_type in ["P", "S"]:
    lines = shared_lines + [
        f"GTFILES {vel_grid_root} {time_grid_root} {wave_type}",
        "GTMODE GRID3D ANGLES_NO",
        "GT_PLFD 1.0e-3 0",  # required: selects the Podvin-Lecomte finite-difference travel-time method
    ]
    path = os.path.join(NLL_DIR, f"grid2time_tvz_{wave_type}.in")
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"Saved: {path}")


In [ ]:
# Build the NLLoc control file. LOCMETH EDT_OT_WT and LOCSEARCH OCT are
# direct matches to the proposal's stated method (time-weighted Equal
# Differential Time likelihood function, Oct-Tree sampling) - not
# substitutions. Search grid mirrors the same extent as the velocity/time
# grids built earlier, for consistency.
NLL_LOC_DIR = os.path.join(NLL_DIR, "loc")
os.makedirs(NLL_LOC_DIR, exist_ok=True)

output_root = os.path.join(NLL_LOC_DIR, "tvz")

nlloc_lines = [
    "CONTROL 1 54321",
    f"TRANS SIMPLE {TARGET_LAT} {TARGET_LON} 0.0",
    "",
    "LOCSIG TVZ earthquake catalogue project - UEA MSc dissertation",
    "LOCCOM EQTransformer picks, PyOcto association, NonLinLoc EDT_OT_WT location",
    f"LOCFILES {obs_path} NLLOC_OBS {time_grid_root} {output_root}",
    "LOCHYPOUT SAVE_NLLOC_ALL FILENAME_PUBLIC_ID",
    "LOCSEARCH OCT 20 20 10 0.01 20000 5000 0 1",
    "LOCGRID 301 301 44 -150.0 -150.0 -3.0 1.0 1.0 1.0 PROB_DENSITY SAVE",
    "LOCMETH EDT_OT_WT 9999.0 4 -1 -1 -1 -1 -1 1",
    "LOCGAU 0.2 0.0",
    "LOCPHASEID P P",
    "LOCPHASEID S S",
    "LOCQUAL2ERR 0.1 0.5 1.0 2.0 99999.9",
    "LOCANGLES ANGLES_NO 5",
] + gtsrce_lines

nlloc_path = os.path.join(NLL_DIR, "nlloc_tvz.in")
with open(nlloc_path, "w") as f:
    f.write("\n".join(nlloc_lines) + "\n")

print(f"Saved: {nlloc_path}")
print("\n".join(nlloc_lines[:15]))
